# Homework №6 | Recover the FM head for Masked Autoregressive Model 

*Partially based on true events*

**Alice** and **Bob** are working on their NeurIPS projects, sharing the same remote server that’s constantly running out of disk space. Since their work overlaps, they both use a common workspace to exchange models and results.

After several days of training, Alice finally obtained a strong **MAR** model. She saved the key checkpoints so she could start finalizing the experiments for her paper.

Later, Bob kicks off his experiments… and quickly fills up the disk with logs and generated samples. To keep things running, he started cleaning old files and checkpoints he assumed were no longer needed. Only afterward he realized that he just deleted Alice’s **best Flow Matching head checkpoint**...

Alice is absolutely crushed. There is no time to retrain, and she needs that checkpoint now. Bob now has to somehow recover it before she missed the NeurIPS deadline.
 
**Task:**  You are Bob (sorry). Recover Alice’s Flow Matching head checkpoint ASAP.


<div>
    <br>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw6/alice_bob_mar.jpg" width=700>
<div>

---

In this task, Bob works with **MAR** (Masked Autoregressive model) that combines:
1. A **masked autoregressive (AR) backbone** — a [ViT](https://arxiv.org/abs/2010.11929) encoder/decoder that processes image patch tokens
2. A **flow matching head** — a small MLP that predicts velocity to generate continuous VAE latent patches

Bob works with the **frozen the pretrained AR backbone** and will build + train the flow matching head from scratch.

**Reference**: [Autoregressive Image Generation without Vector Quantization](https://arxiv.org/abs/2406.11838)

In [ ]:
import os, sys, copy, math
import numpy as np
from tqdm.auto import tqdm
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision.utils import make_grid

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import clear_output

torch.set_num_threads(16)

from models import mar as mar_module
from models.mar import MAR, mask_by_order
from models.vae import AutoencoderKL
from util.crop import center_crop_arr
from util import download

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(42); np.random.seed(42)

---
## Dataset

Due to time constraints, Bob will tune the Flow Matching head using an ImageNet subset of 10 classes. 
The backbone, however, is pretrained on the full ImageNet dataset (1000 classes).

In [ ]:
!wget https://storage.yandexcloud.net/yandex-research/visual-genai/imagenet10.tar
!tar -xf imagenet10.tar

In [ ]:
IMAGENET10_PATH = 'data/imagenet10'

CLASS_META = {
    88:  ('n01818515', 'macaw'),
    130: ('n02007558', 'flamingo'),
    207: ('n02099601', 'golden retriever'),
    281: ('n02123045', 'tabby cat'),
    323: ('n02279972', 'monarch butterfly'),
    388: ('n02510455', 'panda'),
    417: ('n02782093', 'balloon'),
    812: ('n04266014', 'space shuttle'),
    933: ('n07697313', 'cheeseburger'),
    980: ('n09472597', 'volcano'),
}
SELECTED_CLASSES = sorted(CLASS_META.keys())
CLASS_NAMES = {k: v[1] for k, v in CLASS_META.items()}

local_wnids = sorted(wnid for _, (wnid, _) in CLASS_META.items())
wnid_to_inet = {wnid: inet for inet, (wnid, _) in CLASS_META.items()}
LOCAL_TO_IMAGENET = {i: wnid_to_inet[w] for i, w in enumerate(local_wnids)}

def local_labels_to_imagenet(labels):
    return torch.tensor([LOCAL_TO_IMAGENET[l.item()] for l in labels], device=labels.device)

transform_train = transforms.Compose([
    transforms.Lambda(lambda img: center_crop_arr(img, 256)),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)
])
train_dataset = datasets.ImageFolder(os.path.join(IMAGENET10_PATH, 'train'), transform=transform_train)
print(f'Dataset: {len(train_dataset)} images, {len(train_dataset.classes)} classes')

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

In [ ]:
def show_images(images, title='', nrow=8):
    grid = make_grid(images, nrow=nrow, normalize=True, value_range=(-1, 1))
    plt.figure(figsize=(nrow*2, (len(images)//nrow+1)*2))
    plt.imshow(grid.permute(1,2,0).cpu().numpy())
    plt.title(title) 
    plt.axis('off') 
    plt.tight_layout() 
    plt.show()

sample_batch, _ = next(iter(train_loader))
show_images(sample_batch[:16], 'Real images')

---
## How MAR works

```
Image -> [VAE] -> 16x16x16 latents -> [Patchify] -> 256 tokens (16-dim)
  -> [Mask] -> visible tokens + class emb
  -> [Encoder: 12 Transformer blocks]  <- pretrained, frozen
  -> [Decoder: 12 Transformer blocks]  <- pretrained, frozen
  -> z (256 positions, 768-dim)
  -> [FlowLoss: MLP velocity predictor] <- what Bob builds
  -> predicted patch embeddings
```

**Encoder** receives only **visible (unmasked)** tokens — masked are dropped before the Encoder. A buffer of 64 class-embedding tokens is concatenated to input tokens.

**Decoder** pads mask tokens back, applies the transformer blocks and returns condition embeddings `z` at each position.

**Training vs. generation masking**:
- *Training*: single random mask (ratio from truncated Gaussian, 70-100%)
- *Generation*: starts fully masked, iteratively unmasks in the predefined `random order` using the cosine schedule `mask_ratio` $\tau = \cos(\frac{\pi}{2} \cdot \frac{i}{N})$.

**FlowLoss conditioning via AdaLN**: decoder output `z` and timestep `t` are projected and summed: `y = t_emb + z_emb`. Each residual block uses `y` to produce (shift, scale, gate) that modulate layer-normed activations — same mechanism as DiT.

---

**Schematic illustration of the training step**

<div>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw6/mar_training_forward.jpg" width="1000"/>
</div>

---
## Download and create the MAR backbone

Bob has to recover the head for the `MAR-B` backbone of 172M parameters.

The backbone operates in the VAE latent space of $[16\times16\times16]$.

In [ ]:
download.download_pretrained_vae()
download.download_pretrained_marb()

# If Bob has resources and more time, he can try larger models MAR-L or MAR-H 
# However, Alice did not observe significant improvements, though Bob might. 
# download.download_pretrained_marl()
# download.download_pretrained_marh()

In [ ]:
vae = AutoencoderKL(embed_dim=16, ch_mult=(1,1,2,2,4), ckpt_path='pretrained_models/vae/kl16.ckpt').to(device).eval()
for p in vae.parameters(): p.requires_grad = False

def load_pretrained_mar_base():
    model = mar_module.mar_base(buffer_size=64, label_drop_prob=0.1, attn_dropout=0.1, proj_dropout=0.1).to(device)
    ckpt = torch.load('pretrained_models/mar/mar_base/checkpoint-last.pth', map_location='cpu')
    state_dict = {k: v for k, v in ckpt['model_ema'].items() if not k.startswith('diffloss.')}
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    print(f'MAR-Base backbone: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params')
    return model

In [ ]:
def freeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = False

backbone = load_pretrained_mar_base()
freeze_backbone(backbone)
print(f'Frozen: {sum(p.numel() for p in backbone.parameters()) / 1e6:.1f}M')

### Implement `patchify` and `unpatchify` (1 pt)

The MAR backbone operates on **sequences of patch tokens**. Bob needs to implement two functions that convert between the VAE spatial latent and the flat token sequence used by the transformer.

**`patchify(x, patch_size)`** — convert spatial VAE latents to a sequence of patch tokens.
- **Input**: `x` of shape `[B, C, H, W]` — VAE latent (e.g. `[B, 16, 16, 16]`)
- **Output**: `[B, L, D]` — token sequence (e.g. `[B, 256, 16]` when `patch_size=1`)
- Split the spatial dimensions into non-overlapping patches of size `p x p`, then rearrange so each patch becomes one token in a flat sequence. Token dimension `D = C * p^2`.
- **Hint**: use `reshape` + `torch.einsum` for clean dimension permutation.

**`unpatchify(x, patch_size, vae_embed_dim, seq_h, seq_w)`** — inverse of patchify.
- **Input**: `x` of shape `[B, L, D]` — token sequence
- **Output**: `[B, C, H, W]` — spatial VAE latent format
- Reverse the patchify operation: unflatten tokens back into spatial patches and merge them.

In [ ]:
def patchify(x, patch_size):
    """Convert spatial VAE latents [B, C, H, W] to patch token sequence [B, L, D]."""
    # <YOUR CODE>
    pass


def unpatchify(x, patch_size, vae_embed_dim, seq_h, seq_w):
    """Convert patch token sequence [B, L, D] back to spatial VAE latents [B, C, H, W]."""
    # <YOUR CODE>
    pass

# Bind them to the MAR class so existing code works with `model.patchify(x)` / `model.unpatchify(x)`.
MAR.patchify = lambda self, x: patchify(x, self.patch_size)
MAR.unpatchify = lambda self, x: unpatchify(x, self.patch_size, self.vae_embed_dim, self.seq_h, self.seq_w)

In [ ]:
# --- Test patchify / unpatchify shapes ---

# patch_size=1
_x = torch.randn(2, 16, 16, 16)
_tokens = patchify(_x, patch_size=1)
assert _tokens.shape == (2, 256, 16), f"patchify shape: expected (2, 256, 16), got {_tokens.shape}"
_recon = unpatchify(_tokens, patch_size=1, vae_embed_dim=16, seq_h=16, seq_w=16)
assert _recon.shape == (2, 16, 16, 16), f"unpatchify shape: expected (2, 16, 16, 16), got {_recon.shape}"
assert torch.allclose(_x, _recon), "Roundtrip patchify -> unpatchify failed: values differ"

# patch_size=2
_x2 = torch.randn(2, 16, 16, 16)
_t2 = patchify(_x2, patch_size=2)
assert _t2.shape == (2, 64, 64), f"patchify(p=2) shape: expected (2, 64, 64), got {_t2.shape}"
_r2 = unpatchify(_t2, patch_size=2, vae_embed_dim=16, seq_h=8, seq_w=8)
assert torch.allclose(_x2, _r2), "Roundtrip with patch_size=2 failed"
print("patchify/unpatchify tests passed!")

## Implement the FlowLoss head (3 pt)

**`TimestepEmbedder`** *(provided)* — sinusoidal encoding + MLP for continuous `t`.

**`AdaLNResBlock`** — residual block with Adaptive Layer Normalization:
1. `y` -> MLP (`Activation -> Linear`) -> 3 vectors: (shift, scale, gate)
2. `h = LN(x) * (1 + scale) + shift`
3. `h = MLP(h)` (Two-layer MLP)
4. `output = x + gate * h`

**`FinalLayer`** — final projection with AdaLN (shift + scale only, no gate):
1. `y` -> MLP (`Activation -> Linear`) -> 2 vectors: (shift, scale)
2. `x = LN(x, elementwise_affine=False) * (1 + scale) + shift`
3. `output = Linear(x)`

**`VelocityMLP`** — the full denoiser:
1. Project input to `model_channels`, embed timestep and conditioning, sum: `y = t_emb + c_emb`
2. Stack of `AdaLNResBlock`s modulated by `y`
3. `FinalLayer` -> velocity `[N, C]`

**`FlowLoss`** — wraps VelocityMLP:
- `forward(target, z, mask)`: **v-loss** for $\mathbf{x}_t = (1-t) \mathbf{x}_0 + t \epsilon$

In [ ]:
class TimestepEmbedder(nn.Module):
    """
        Sinusoidal encoding + MLP for continuous timestep t
    """
    def __init__(self, hidden_size, freq_dim=256):
        super().__init__()
        self.freq_dim = freq_dim
        self.mlp = nn.Sequential(
            nn.Linear(freq_dim, hidden_size), 
            nn.SiLU(), 
            nn.Linear(hidden_size, hidden_size)
        )
        self._init_weights()
        
    def _init_weights(self):
        nn.init.normal_(self.mlp[0].weight, std=0.02)
        nn.init.normal_(self.mlp[2].weight, std=0.02)
        
    def forward(self, t):
        half = self.freq_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32) / half).to(t.device)
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if self.freq_dim % 2:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
        return self.mlp(emb)


class AdaLNResBlock(nn.Module):
    """
        y -> (shift, scale, gate);
        h = LN(x)*(1+scale)+shift; 
        out = x + gate*MLP(h)
    """
    def __init__(self, channels):
        super().__init__()
        # <YOUR CODE>

    def forward(self, x, y):
        # <YOUR CODE>
        pass


class FinalLayer(nn.Module):
    """
        y -> (shift, scale); 
        x = LN(x, no_affine)*(1+scale)+shift; 
        out = Linear(x)
    """
    def __init__(self, model_channels, out_channels):
        super().__init__()
        # <YOUR CODE>

    def forward(self, x, y):
        # <YOUR CODE>
        pass


class VelocityMLP(nn.Module):
    def __init__(self, in_channels, model_channels, z_channels, num_blocks):
        """
            t_embed, c_proj, x_proj, blocks (AdaLNResBlock), out (FinalLayer).
        """
        super().__init__()
        
        # <YOUR CODE>
        
    def _init_weights(self):
        """ 
            Zero-init all adaLN **outputs** and the final linear layer.
            Remember to call it in __init__
        """
        # <YOUR CODE>
        pass

    def forward(self, x, t, c):
        """
            y = t_embed(t)+c_proj(c) -> blocks -> out.
        """
        # <YOUR CODE>
        pass

    def cfg_forward(self, x, t, c, cfg_scale):
        # <YOUR CODE>
        pass

In [ ]:
class FlowLoss(nn.Module):
    def __init__(self, token_dim, z_dim, depth, width, num_steps=50):
        super().__init__()
        self.token_dim = token_dim
        self.net = VelocityMLP(token_dim, width, z_dim, depth)
        self.num_steps = num_steps
        
    def sample_t(self, n: int, device=None):
        """ 
            Implement logit normal timestep sampling distribution. 
            You can use it without shift since the dim is small
        """
        # <YOUR CODE>
        pass

    def forward(self, target, z, mask=None):
        # You can adopt your implementation from HW3
        # <YOUR CODE>
        pass


# Quick test to debug shapes. Reference loss values are in [1.5, 3.0]
_fl = FlowLoss(16, 768, 2, 256, 10).to(device)
print(f'FlowLoss test: loss={_fl(torch.randn(4,16,device=device), torch.randn(4,768,device=device), torch.ones(4,device=device)).item():.4f}')
del _fl

In [ ]:
@torch.no_grad()
def sample_loop(net, noise, num_steps, z, cfg=1.0):
    """
        Euler ODE t=1->0
        
        noise is x_T
        
        You can adopt your implementation from HW3
    """
    # <YOUR CODE>
    pass

### Implement EMA update

It is often beneficial to maintain an **Exponential Moving Average (EMA)** of model weights during training. The EMA model produces smoother, higher-quality outputs at inference time.

- `create_ema(model)`: deep-copy the model and freeze it
- `ema_update(ema_model, model, decay)`: $\theta_{ema} = decay \cdot \theta_{ema} + (1 - decay) \cdot \theta$


In [ ]:
@torch.no_grad()
def ema_update(ema_model, model, decay=0.992):
    # <YOUR CODE>
    pass

def create_ema(model):
    # <YOUR CODE>
    pass

## Implement `sample_tokens` (iterative unmasking) (4 pt)

Implement `sample_tokens(model, bsz, ...)` — the iterative masked autoregressive generation loop.

**High-level algorithm**:

1. **Initialize** all tokens as masked (`mask = ones`), with an empty token buffer (`tokens = zeros`). Sample random generation orders via `model.sample_orders(bsz)`.

2. **Iterate** for `num_iter` steps. At each step:
   - Get class embeddings. (`model.fake_latent` - unconditional embeddings for CFG) 
   - **Encode + decode** using the frozen backbone to get conditioning `z` at all positions. `model.forward_mae_encoder(tokens, mask, class_embedding)` and `model.forward_mae_decoder(enc_out, mask)`
   - **Cosine schedule**: compute $\tau = \cos(\frac{\pi}{2} \cdot \frac{i}{N})$ to determine the ratio of tokens to keep *masked* at $i$-th step out of $N$. Get the new mask via `mask_by_order(mask_len, orders, bsz, seq_len)`.
   - **Identify newly unmasked positions**. At the last step, predict all remaining masked positions.
   - **Sample** new tokens at those positions using `sample_loop` implemented above.
   - **Update** the token buffer with sampled values.

3. **Return** Spatial latents `[B, C, H, W]`.


### CFG Strategies

In this task, Bob also has to implement three CFG strategies for MAR sampling (not for the FM head):

* `'constant'`: classical CFG
* `'dynamic'`: increases from 1 to `cfg` proportionally to the fraction of unmasked tokens $1-\tau$
* `'interval'`: 1 for first $K$ steps and constant `cfg` elsewhere

---

**Schematic illustration of one intermediate sampling step**

<div>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw6/mar_sampling.jpg" width="1000"/>
</div>

In [ ]:
def sample_tokens(
    model, 
    bsz, 
    num_iter=64, 
    cfg=4.0, 
    cfg_schedule='constant',      
    labels=None, 
    flow_loss=None, 
    cfg_off_steps=None # first K steps to turn off CFG for the interval strategy  
):
    """ Do not forget about three CFG strategies """
    # <YOUR CODE>
    pass

In [ ]:
# --- Test sample_tokens shapes ---
_fl_test = FlowLoss(backbone.token_embed_dim, backbone.decoder_embed.out_features, depth=2, width=256, num_steps=2).to(device)

with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    _labels = torch.tensor([88, 130], device=device)
    
    # Test with CFG
    _out = sample_tokens(backbone, bsz=2, num_iter=1, cfg=4.0, labels=_labels, flow_loss=_fl_test)
    assert _out.shape == (2, 16, 16, 16), f"sample_tokens CFG shape: expected (2, 16, 16, 16), got {_out.shape}"
    
    # Test without CFG
    _out2 = sample_tokens(backbone, bsz=2, num_iter=1, cfg=1.0, labels=_labels, flow_loss=_fl_test)
    assert _out2.shape == (2, 16, 16, 16), f"sample_tokens no-CFG shape: expected (2, 16, 16, 16), got {_out2.shape}"

del _fl_test
print("sample_tokens tests passed!")

### Generate images

In [ ]:
@torch.no_grad()
def generate(
        model, flow_loss, labels, num_iter=64, cfg=4.0, 
        cfg_schedule='constant', cfg_off_steps=None, seed=42
    ):
    model.eval(); flow_loss.eval()
    torch.manual_seed(seed); np.random.seed(seed)
    with torch.autocast('cuda', dtype=torch.float16):
        tokens = sample_tokens(model, len(labels), num_iter=num_iter, cfg=cfg, cfg_schedule=cfg_schedule,
                               labels=labels, flow_loss=flow_loss, cfg_off_steps=cfg_off_steps)
        images = vae.decode(tokens / 0.2325).float()
    return images

### Implement the MAR forward pass (1 pt)

Implement `mar_forward_pass(model, flow_loss, x, labels)` — the full MAR forward pass used during training:
1. `model.class_emb(labels)` — class embedding
2. `model.patchify(x)` → tokens
3. `model.sample_orders(bsz)` → orders; `model.random_masking(tokens, orders)` → mask
4. `model.forward_mae_encoder(tokens, mask, class_embedding)` → encoder output
5. `model.forward_mae_decoder(enc_out, mask)` → decoder conditioning `z`
6. `flow_loss(target, z, mask)` → loss

Returns the final `loss` for FlowLoss.

In [ ]:
def mar_forward_pass(model, flow_loss, x, labels):
    """
        Full MAR forward: patchify -> mask -> encoder -> decoder -> flow_loss.
        Returns loss.
    """
    # <YOUR CODE>
    pass

In [ ]:
# --- Test mar_forward_pass ---
_fl_test = FlowLoss(backbone.token_embed_dim, backbone.decoder_embed.out_features, depth=2, width=256, num_steps=10).to(device)
_x_test = torch.randn(2, 16, 16, 16, device=device)
_labels_test = torch.tensor([88, 130], device=device)
with torch.autocast('cuda', dtype=torch.float16):
    _loss = mar_forward_pass(backbone, _fl_test, _x_test, _labels_test)
assert _loss.dim() == 0, f"Loss should be scalar, got shape {_loss.shape}"
assert torch.isfinite(_loss), f"Loss is not finite: {_loss.item()}"
assert _loss.requires_grad, "Loss should have grad_fn for backprop"
del _fl_test
print(f"mar_forward_pass test passed! loss={_loss.item():.4f}")

### Training

Uses `mar_forward_pass` and `ema_update` you implemented above.

**Bob uses the Alice's training code where the mixed-precision training (FP16/FP32) is already added**

In [ ]:
def train_model(model, flow_loss, ema_flow_loss, optimizer, max_iters):
    model.eval(); flow_loss.train()
    losses = []
    loss_sum = loss_cnt = iteration = 0
    pbar = tqdm(total=max_iters, desc='Training')

    scaler = torch.amp.GradScaler("cuda", enabled=True)
        
    while True:
        for images, local_labels in train_loader:
            iteration += 1
            images = images.to(device, non_blocking=True)
            labels = local_labels_to_imagenet(local_labels.to(device))
            with torch.no_grad():
                x = vae.encode(images).sample().mul_(0.2325)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = mar_forward_pass(model, flow_loss, x, labels)
            
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            ema_update(ema_flow_loss, flow_loss, decay=0.992)
            
            loss_sum += loss.item(); loss_cnt += 1
            pbar.update(1)
            pbar.set_postfix(loss=f'{loss.item():.4f}')

            if iteration % 100 == 0:
                ema_flow_loss.eval()
                with torch.no_grad():
                    demo = generate(model, ema_flow_loss, torch.tensor(SELECTED_CLASSES, device=device))
                    demo = ((demo + 1) / 2).clamp(0, 1).cpu()
                flow_loss.train()
                losses.append(loss_sum / loss_cnt); loss_sum = loss_cnt = 0
                clear_output(wait=True)
                n_cls = len(SELECTED_CLASSES)
                fig = plt.figure(figsize=(14, 12), constrained_layout=True)
                gs = gridspec.GridSpec(3, 5, figure=fig, height_ratios=[2, 1, 1])
                ax = fig.add_subplot(gs[0, :])
                ax.semilogy(100 * torch.arange(1, len(losses)+1), losses)
                ax.set_title('Loss', fontsize=14); ax.set_xlabel('Iteration', fontsize=12); ax.set_ylabel('Loss', fontsize=12); ax.grid()
                for idx in range(n_cls):
                    row, col = 1 + idx // 5, idx % 5
                    a = fig.add_subplot(gs[row, col])
                    a.imshow(demo[idx].permute(1,2,0).numpy())
                    a.axis('off'); a.set_title(CLASS_NAMES[SELECTED_CLASSES[idx]][:10], fontsize=9)
                plt.show()
                pbar = tqdm(total=max_iters, initial=iteration, desc='Training')
            if iteration >= max_iters: break
        if iteration >= max_iters: break
    pbar.close()

In [ ]:
flow_loss = FlowLoss(
    token_dim=backbone.token_embed_dim, z_dim=backbone.decoder_embed.out_features,
    depth=6, width=1024, num_steps=50
).to(device)
ema_flow_loss = create_ema(flow_loss)
optimizer = torch.optim.AdamW(flow_loss.parameters(), lr=1e-4, betas=(0.9, 0.95))
print(f'FlowLoss: {sum(p.numel() for p in flow_loss.parameters()) / 1e6:.1f}M params')

# In the Alice's implementation, the head contains 35.7M parameters 
# but she will not care much if Bob makes it slightly larger or smaller 

train_model(backbone, flow_loss, ema_flow_loss, optimizer, max_iters=3000)

# If Bob has resources and time and wants to improve the results,
# he can consider larger models MAR-L or MAR-H (see the download cell in the beginning)
# Increase training iters and batch size. 
# Head is already large enough but he may also try to play with depth and width but needs more training iterations.

# Important: Alice did not notice significant gains from scaling. 
# However, Alice would be happy if Bob can further improve it!   

### Alice's results for MAR-B / 3K iterations / batch size 16 / lr=2e-4

<div>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw6/reference_mar_loss.jpg" width=800>
<div>

---
## Tuning Sampling Hyperparameters (1 pt)

To make up for his mistake, Bob agreed to help Alice tune the sampling hyperparameters.

Run the following three experiments:

1) Find a better `num_steps` value for the FM head.
2) Find a better `num_iter` value for AR sampling.
3) Find a better `cfg_strategy` and `cfg_scale` setting for AR sampling.

---
###  FM head steps

In [ ]:
steps_to_try = [2, 4, 6, 8, 10, 15, 25, 50]
demo_labels = torch.tensor([88, 207, 323, 388, 933], device=device)
all_images = []
for ns in steps_to_try:
    ema_flow_loss.num_steps = ns
    all_images.append(generate(backbone, ema_flow_loss, demo_labels, seed=0))
    
fig, axes = plt.subplots(len(steps_to_try), len(demo_labels), figsize=(3*len(demo_labels), 3*len(steps_to_try)))
for row, (ns, imgs) in enumerate(zip(steps_to_try, all_images)):
    for col in range(len(demo_labels)):
        axes[row, col].imshow(((imgs[col].permute(1,2,0).cpu().numpy()+1)/2).clip(0,1))
        axes[row, col].axis('off')
        if col == 0: 
            axes[row, col].set_ylabel(f'{ns} steps', fontsize=14, rotation=0, labelpad=70)
        if row == 0: 
            axes[row, col].set_title(CLASS_NAMES[demo_labels[col].item()], fontsize=12)
plt.suptitle('Varying Euler Steps (cfg=4.0)', fontsize=16); plt.tight_layout(); plt.show()

**< TODO: Bob's best number of the FM head steps >**

In [ ]:
ema_flow_loss.num_steps = # <Bob's BEST num_steps>

---
### AR iterations

In [ ]:
iters_to_try = [1, 4, 16, 64, 128, 256]
demo_labels = torch.tensor([88, 207, 323, 388, 933], device=device)
all_images = []
for ni in iters_to_try:
    all_images.append(generate(backbone, ema_flow_loss, demo_labels, num_iter=ni, seed=0))
    
fig, axes = plt.subplots(len(iters_to_try), len(demo_labels), figsize=(3*len(demo_labels), 3*len(iters_to_try)))
for row, (ni, imgs) in enumerate(zip(iters_to_try, all_images)):
    for col in range(len(demo_labels)):
        axes[row, col].imshow(((imgs[col].permute(1,2,0).cpu().numpy()+1)/2).clip(0,1))
        axes[row, col].axis('off')
        if col == 0: 
            axes[row, col].set_ylabel(f'{ni} iters', fontsize=14, rotation=0, labelpad=70)
        if row == 0: 
            axes[row, col].set_title(CLASS_NAMES[demo_labels[col].item()], fontsize=12)
plt.suptitle('Varying AR Iterations (cfg=4.0)', fontsize=16); plt.tight_layout(); plt.show()

**< TODO: Bob's best number of AR iterations to achieve better speed-quality tradeoff >**

---
### CFG strategies

1. **Dynamic** — cfg increases with unmasked fraction
2. **Constant** — same cfg throughout (default)
3. **Interval** — turn off CFG for the first $K$ MAR steps. Then, use constant CFG.

Try different `cfg_scale`

In [ ]:
demo_labels = torch.tensor([88, 207, 323, 388, 933], device=device)
configs = [
    ('dynamic', 4.0, None),
    ('constant', 4.0, None),
    ('interval', 4.0, 4),
    ('interval', 4.0, 8),
    ('interval', 4.0, 16),
]

all_images = []
for sched, cfg_val, K in configs:
    all_images.append(
        generate(
            backbone, ema_flow_loss, demo_labels, cfg=cfg_val, 
            cfg_schedule=sched, cfg_off_steps=K, seed=0
        )
    )

fig, axes = plt.subplots(len(configs), len(demo_labels), figsize=(3*len(demo_labels), 3*len(configs)))
for row, ((label, *_), imgs) in enumerate(zip(configs, all_images)):
    for col in range(len(demo_labels)):
        axes[row, col].imshow(((imgs[col].permute(1,2,0).cpu().numpy()+1)/2).clip(0,1))
        axes[row, col].axis('off')
        if col == 0: 
            axes[row, col].set_ylabel(label, fontsize=11, rotation=0, labelpad=110)
        if row == 0: 
            axes[row, col].set_title(CLASS_NAMES[demo_labels[col].item()], fontsize=12)
plt.suptitle('CFG Strategies Comparison', fontsize=16); plt.tight_layout(); plt.show()

**<TODO: Bob's best CFG strategy and CFG scale >**



---

**Yeah! You did it! Alice can still make the NeurIPS deadline! It was also so nice of Bob to tune the hyperparameters for her <3**

<div>
    <br>
<img src="https://storage.yandexcloud.net/yandex-research/visual-genai/hw6/mar_final_image_alice_bob.jpg" width=700>
<div>
